# Regression vs Classification Strategies: Linear, Lasso, Logistic & Decision Trees

Every model sees the same 1/5/15/20-day momentum features of S&P 500 stocks, on the same date-based train/test split. The **regressions** forecast the next-day *return*; the **classifiers** forecast the *probability* it is positive. Each model becomes a long/flat strategy, and **all of them are compared against the equal-weighted benchmark in one Plotly dashboard**.

| Model | Predicts | Long when |
|---|---|---|
| Linear regression (OLS) | next-day return | forecast > 0 |
| Lasso (alpha by time-series CV) | next-day return | forecast > 0 |
| Logistic regression | P(next-day return > 0) | P > 0.5 |
| Decision tree (default) | P(up) | P > 0.5 |
| Decision tree (tuned with time-ordered CV) | P(up) | P > 0.5 |

**Everything shared comes from `func_lib.py`:**
- Data: `createHistPrices`, `computingReturns`
- Split, scaling and CV folds: `chronological_split`, `scale_features`, `date_based_cv_splits`
- Backtests: `build_pred_frame`, `compute_BM_Perf`, `compute_strat_perf` (with `plot=False`)
- Comparison: `strategy_comparison_figure`, `strategy_stats_table`, `stats_table_figure`, coefficient charts
- The dark Plotly theme

In [ ]:
"""
Imports: statsmodels for linear and logistic regression with full statistics, scikit-learn for
Lasso and decision trees, and func_lib for the data, split, backtest and chart helpers.
"""
import os
import sys

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

# Make func_lib importable from this folder or its parent, without duplicate sys.path entries
for path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))):
    if os.path.exists(os.path.join(path, 'func_lib.py')) and path not in sys.path:
        sys.path.append(path)

import func_lib   # importing it also applies the dark Plotly theme ('quant_dark')
from func_lib import build_pred_frame

In [ ]:
"""
Settings for the whole notebook: date range, momentum windows used as features, the train/test
split, the thresholds that turn forecasts into long/flat positions, and the Lasso alpha grid.
"""
# ---- Configuration ----
START_DATE, END_DATE = '2000-01-01', '2026-09-23'
MOMENTUM_WINDOWS = [1, 5, 15, 20]
TRAIN_FRACTION = 0.7       # first 70% of dates for training, rest for testing
PROB_THRESHOLD = 0.5       # classifiers: long when P(up) > threshold
RETURN_THRESHOLD = 0.0     # regressions: long when forecast return > threshold
CV_SPLITS = 5
RANDOM_STATE = 42

USE_LASSO_CV = True        # choose Lasso alpha by time-series CV
FIXED_ALPHA = 1e-5         # used when USE_LASSO_CV is False
ALPHA_GRID = np.logspace(-7, -2, 30)

FEATURES = [f'{m}_d_returns' for m in MOMENTUM_WINDOWS]
RETURN_COL = 'F_1_d_returns'        # regression target and realized forward return
DIRECTION_COL = 'F_1_d_returns_Ind' # classification target: 1 if forward return > 0

## 1. Data and one shared chronological split

In [ ]:
"""
Downloads S&P 500 prices, builds 1/5/15/20-day momentum features and the next-day return, and
makes one chronological split shared by every model. Features are scaled on training data only,
and cross-validation folds are cut by date so no model trains on the future.
"""
historical_prices = func_lib.createHistPrices(START_DATE, END_DATE)
total_returns = func_lib.computingReturns(historical_prices, MOMENTUM_WINDOWS)
total_returns[DIRECTION_COL] = (total_returns[RETURN_COL] > 0).astype(int)

split = func_lib.chronological_split(total_returns, FEATURES, RETURN_COL, RETURN_COL, TRAIN_FRACTION)
x_train, x_test = split['x_train'], split['x_test']
y_ret_train, y_ret_test = split['y_train'], split['y_test']                 # regression targets
y_dir_train = total_returns.loc[x_train.index, DIRECTION_COL]               # classification targets
y_dir_test = total_returns.loc[x_test.index, DIRECTION_COL]
test_fwd_returns = split['fwd_returns_test']
x_train_scaled, x_test_scaled, scaler = func_lib.scale_features(x_train, x_test)
cv_folds = func_lib.date_based_cv_splits(x_train_scaled.index, CV_SPLITS)

print(f"Split date: {split['split_date']} | train rows: {len(x_train):,} | test rows: {len(x_test):,}")
print(f'Share of up days  train: {y_dir_train.mean():.3f} | test: {y_dir_test.mean():.3f}')
total_returns.head()

## 2. Regression models: forecast the next-day return

In [ ]:
"""
Linear regression (OLS) of next-day return on the scaled momentum features. The summary table
shows each coefficient with its standard error and p-value.
"""
x_train_const = sm.add_constant(x_train_scaled)
x_test_const = sm.add_constant(x_test_scaled)

# Linear regression (OLS): statsmodels for the coefficient table with standard errors and p-values
ols_model = sm.OLS(y_ret_train, x_train_const).fit()
print(ols_model.summary())

In [ ]:
"""
Lasso regression: like OLS but with an L1 penalty that shrinks weak coefficients to exactly zero.
The penalty strength (alpha) is chosen by date-ordered cross-validation. Compares the two sets
of coefficients side by side.
"""
# Lasso: L1 shrinkage, with alpha chosen on date-ordered folds
if USE_LASSO_CV:
    lasso_model = LassoCV(alphas=ALPHA_GRID, cv=cv_folds, n_jobs=-1).fit(x_train_scaled, y_ret_train)
    print(f'CV-selected alpha: {lasso_model.alpha_:.2e}')
else:
    lasso_model = Lasso(alpha=FIXED_ALPHA).fit(x_train_scaled, y_ret_train)

coefficients = pd.DataFrame({'Linear (OLS)': ols_model.params[FEATURES],
                             'Lasso': pd.Series(lasso_model.coef_, index=FEATURES)})
print(coefficients.assign(lasso_selected=coefficients['Lasso'] != 0))
func_lib.coefficient_comparison_figure(coefficients, 'Linear vs Lasso coefficients (scaled features)').show()

## 3. Classification models: forecast P(next-day return > 0)

In [ ]:
"""
Logistic regression: predicts the probability that tomorrow's return is positive.
Prints the coefficient table and charts the coefficients (in log-odds).
"""
logit_model = sm.Logit(y_dir_train, x_train_const).fit(disp=False)
print(logit_model.summary())
func_lib.coefficient_bar_figure(logit_model.params[FEATURES], 'Logistic regression coefficients (log-odds)').show()

In [ ]:
"""
Two decision trees: an unpruned one, which overfits noisy daily returns and serves as a baseline,
and a tuned one whose depth and leaf sizes are chosen by grid search on date-ordered folds.
"""
# Default tree: an unpruned tree overfits noisy daily returns, so it serves as a baseline
tree_model = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(x_train_scaled, y_dir_train)

# Tuned tree: grid search on date-ordered expanding-window folds (no look-ahead)
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid={'criterion': ['gini'], 'max_depth': [None, 5, 10, 15],
                'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]},
    cv=cv_folds, scoring='roc_auc', n_jobs=-1,
).fit(x_train_scaled, y_dir_train)
print('Best params:', grid_search.best_params_, f'| CV ROC-AUC: {grid_search.best_score_:.4f}')

## 4. Out-of-sample predictions and predictive accuracy
Each model produces a score: a forecast return or a probability.
- **Hit rate:** how often the long/flat call matched the actual direction.
- **ROC-AUC:** how well the score *ranks* up days above down days (0.5 = no skill).
- **Rank IC:** the Spearman correlation between the score and the realized return.

In [ ]:
"""
Out-of-sample predictions for all five models, scored on the test period: hit rate, ROC-AUC,
rank correlation and the share of time each would be long. An AUC of 0.5 means no skill.
"""
MODELS = {   # name -> (out-of-sample score, kind)
    'Linear':    (ols_model.predict(x_test_const), 'regression'),
    'Lasso':     (lasso_model.predict(x_test_scaled), 'regression'),
    'Logistic':  (logit_model.predict(x_test_const), 'classifier'),
    'DecTree':   (tree_model.predict_proba(x_test_scaled)[:, 1], 'classifier'),
    'DecTreeCV': (grid_search.best_estimator_.predict_proba(x_test_scaled)[:, 1], 'classifier'),
}
THRESHOLDS = {'regression': RETURN_THRESHOLD, 'classifier': PROB_THRESHOLD}

accuracy = {}
for name, (score, kind) in MODELS.items():
    score = pd.Series(np.asarray(score).ravel(), index=x_test.index)
    long = score > THRESHOLDS[kind]
    accuracy[name] = {'Type': kind.title(),
                      'Hit Rate': accuracy_score(y_dir_test, long),
                      'ROC-AUC': roc_auc_score(y_dir_test, score),
                      'Rank IC': (score.corr(test_fwd_returns, method='spearman')
                                  if score.nunique() > 1 else np.nan),   # constant score -> undefined
                      'Time Long': long.mean()}
accuracy = pd.DataFrame(accuracy)
func_lib.stats_table_figure(accuracy, 'Out-of-sample predictive accuracy', index_name='Metric',
                            pct_rows=('Hit Rate', 'Time Long'), decimals=4).show()

## 5. Backtest every strategy against the benchmark
Each strategy is equal-weighted across the stocks it is long each day, and flat otherwise. The benchmark is the equal-weighted average of all stocks. `plot=False` skips the per-model charts, so one dashboard compares everything.

In [ ]:
"""
Backtest. Each model becomes a long/flat strategy (regressions go long when the forecast is
positive, classifiers when P(up) > 50%) and is compared with the equal-weighted benchmark.
"""
def regression_strategy(forecast):
    """Long (1) when the forecast return beats the threshold, otherwise flat (0)."""
    return int(forecast > RETURN_THRESHOLD)


def classifier_strategy(prob_up):
    """Long (1) when P(up) beats the threshold, otherwise flat (0)."""
    return int(prob_up > PROB_THRESHOLD)


STRATEGIES = {'regression': regression_strategy, 'classifier': classifier_strategy}

first_name = next(iter(MODELS))
cum_returns, calendar_returns = func_lib.compute_BM_Perf(
    build_pred_frame(MODELS[first_name][0], x_test.index, test_fwd_returns, first_name), plot=False)

for name, (score, kind) in MODELS.items():
    preds = build_pred_frame(score, x_test.index, test_fwd_returns, name)
    cum_returns, calendar_returns = func_lib.compute_strat_perf(
        preds, cum_returns, calendar_returns, STRATEGIES[kind], name, plot=False)

In [ ]:
"""
Chart of every strategy against the benchmark: growth of $1, drawdowns and calendar-year returns.
"""
func_lib.strategy_comparison_figure(
    cum_returns, calendar_returns,
    f"All strategies vs equal-weighted benchmark (test period from {split['split_date']:%Y-%m-%d})",
).show()

In [ ]:
"""
Performance table for every strategy and the benchmark (CAGR, volatility, Sharpe, drawdown,
excess return), and prints which strategy had the highest Sharpe ratio.
"""
comparison = func_lib.strategy_stats_table(cum_returns)
func_lib.stats_table_figure(
    comparison, 'Strategy performance vs benchmark (out-of-sample)', index_name='Metric',
    pct_rows=('Total Return', 'CAGR', 'Annualized Return (arith.)', 'Annualized Volatility',
              'Downside Deviation', 'Max Drawdown', 'Positive Days', 'Excess CAGR vs Benchmark'),
).show()

best = comparison.loc['Sharpe Ratio'].drop('S&P500').idxmax()
print(f"Highest Sharpe: {best} ({comparison.loc['Sharpe Ratio', best]:.2f}) vs benchmark "
      f"{comparison.loc['Sharpe Ratio', 'S&P500']:.2f}")

## 6. Reading the results
- **Regression vs classification:** both families use the same information, so differences mostly come from the loss function. A model that ranks days well (higher AUC / IC) can still lose to the benchmark if it sits flat during strong up-markets. Check **Time Long** next to **Excess CAGR**.
- **Lasso shrinkage:** if CV shrinks every coefficient to zero, the Lasso forecast is a constant, and the strategy is always long (equal to the benchmark) or always flat.
- **Trees:** the unpruned tree is the overfitting baseline. The tuned tree uses date-ordered CV, so its improvement (if any) is honest out-of-sample.
- **Next steps:** add transaction costs on position changes, try a cross-sectional ranking (long the top decile instead of long/flat), and add gradient boosting (Bali et al. found nonlinear models help most).